In [56]:
import re
from collections import Counter

import lorem
from tqdm import tqdm

In byte-pair encoding we build a vocabulary by iteratively merging the most frequent pair of adjacent tokens.

1. Start with individual characters as tokens.
2. Find the most frequent pair of adjacent tokens.
3. Merge that pair into a new token.
4. Repeat until we reach the desired size of our vocabulary.


We start off by creating a small corpus of text.


In [57]:
text = lorem.text()
words = re.findall(r"\w+|[^\w\s]", text)[:10]

In [58]:
words

['Modi',
 'tempora',
 'est',
 'tempora',
 '.',
 'Aliquam',
 'quaerat',
 'dolore',
 'non',
 'adipisci']

We then split the words into a list of characters.


In [59]:
words_char_list = [list(word) for word in words]

In [60]:
words_char_list

[['M', 'o', 'd', 'i'],
 ['t', 'e', 'm', 'p', 'o', 'r', 'a'],
 ['e', 's', 't'],
 ['t', 'e', 'm', 'p', 'o', 'r', 'a'],
 ['.'],
 ['A', 'l', 'i', 'q', 'u', 'a', 'm'],
 ['q', 'u', 'a', 'e', 'r', 'a', 't'],
 ['d', 'o', 'l', 'o', 'r', 'e'],
 ['n', 'o', 'n'],
 ['a', 'd', 'i', 'p', 'i', 's', 'c', 'i']]

These characters are what we use to seed our vocabulary.


In [61]:
vocab = list(set(char for word in words_char_list for char in word))

In [62]:
print(vocab)

['n', 'r', 'a', '.', 'm', 't', 'o', 'e', 'd', 'M', 'p', 'i', 'A', 's', 'u', 'c', 'q', 'l']


The next step is to find the most frequent pair of adjacent characters across all words.


In [63]:
pair_counts = {}
for word in words_char_list:
    for i in range(len(word) - 1):
        pair = (word[i], word[i + 1])
        if pair in pair_counts:
            pair_counts[pair] += 1
        else:
            pair_counts[pair] = 1

In [64]:
pair_counts

{('M', 'o'): 1,
 ('o', 'd'): 1,
 ('d', 'i'): 2,
 ('t', 'e'): 2,
 ('e', 'm'): 2,
 ('m', 'p'): 2,
 ('p', 'o'): 2,
 ('o', 'r'): 3,
 ('r', 'a'): 3,
 ('e', 's'): 1,
 ('s', 't'): 1,
 ('A', 'l'): 1,
 ('l', 'i'): 1,
 ('i', 'q'): 1,
 ('q', 'u'): 2,
 ('u', 'a'): 2,
 ('a', 'm'): 1,
 ('a', 'e'): 1,
 ('e', 'r'): 1,
 ('a', 't'): 1,
 ('d', 'o'): 1,
 ('o', 'l'): 1,
 ('l', 'o'): 1,
 ('r', 'e'): 1,
 ('n', 'o'): 1,
 ('o', 'n'): 1,
 ('a', 'd'): 1,
 ('i', 'p'): 1,
 ('p', 'i'): 1,
 ('i', 's'): 1,
 ('s', 'c'): 1,
 ('c', 'i'): 1}

We could also use the `Counter` class from the `collections` module to count the pairs.

```python
pair_counts = Counter(
    pair for word in words_char_list for pair in zip(word[:-1], word[1:])
)
```


Next we need to pick the pair with the highest count as our first merge.


In [65]:
max_pair = max(pair_counts, key=lambda p: pair_counts[p])

In [66]:
max_pair

('o', 'r')

And merge it into a new token.


In [67]:
new_token = max_pair[0] + max_pair[1]
new_token

'or'

We can now update our vocabulary with the new token.


In [68]:
vocab.append(new_token)

In [69]:
print(vocab)

['n', 'r', 'a', '.', 'm', 't', 'o', 'e', 'd', 'M', 'p', 'i', 'A', 's', 'u', 'c', 'q', 'l', 'or']


We now update our words to use the new token.


In [ ]:
new_words = []
for word in words:
    i = 0
    new_word = []
    while i < len(word):
        if i < len(word) - 1 and (word[i], word[i + 1]) == max_pair:
            new_word.append(new_token)
            i += 2
        else:
            new_word.append(word[i])
            i += 1
    new_words.append(new_word)
words = new_words

We can also do this clever trick for more succinct code.

This works by joining the words with a separator (e.g. `\x00` because it is not a character that will appear in the text), replacing the pair with the new token, and then splitting the words back apart.

```python
sep = "\x00"
pair_str = sep.join(max_pair)
print(pair_str)
print(sep.join(words[0]))
words = [sep.join(w).replace(pair_str, new_token).split(sep) for w in words]
```


In [75]:
words

[['M', 'o', 'd', 'i'],
 ['t', 'e', 'm', 'p', 'or', 'a'],
 ['e', 's', 't'],
 ['t', 'e', 'm', 'p', 'or', 'a'],
 ['.'],
 ['A', 'l', 'i', 'q', 'u', 'a', 'm'],
 ['q', 'u', 'a', 'e', 'r', 'a', 't'],
 ['d', 'o', 'l', 'or', 'e'],
 ['n', 'o', 'n'],
 ['a', 'd', 'i', 'p', 'i', 's', 'c', 'i']]

The next step is to repeat this process until we run out of pairs to merge.


In [78]:
vocab = list(set(char for word in words_char_list for char in word))

num_merges = 100
words = [word[:] for word in words_char_list]  # deep copy to preserve original

for _ in tqdm(range(num_merges)):
    pair_counts = Counter(pair for word in words for pair in zip(word[:-1], word[1:]))
    if not pair_counts:
        break
    max_pair = max(pair_counts, key=lambda p: pair_counts[p])
    new_token = max_pair[0] + max_pair[1]
    vocab.append(new_token)

    new_words = []
    for word in words:
        i = 0
        new_word = []
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == max_pair:
                new_word.append(new_token)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_words.append(new_word)
    words = new_words

 33%|███▎      | 33/100 [00:00<00:00, 63725.61it/s]


In [79]:
print(len(vocab))
vocab

51


['n',
 'r',
 'a',
 '.',
 'm',
 't',
 'o',
 'e',
 'd',
 'M',
 'p',
 'i',
 'A',
 's',
 'u',
 'c',
 'q',
 'l',
 'or',
 'di',
 'te',
 'tem',
 'temp',
 'tempor',
 'tempora',
 'qu',
 'qua',
 'Mo',
 'Modi',
 'es',
 'est',
 'Al',
 'Ali',
 'Aliqua',
 'Aliquam',
 'quae',
 'quaer',
 'quaera',
 'quaerat',
 'do',
 'dol',
 'dolor',
 'dolore',
 'no',
 'non',
 'adi',
 'adip',
 'adipi',
 'adipis',
 'adipisc',
 'adipisci']